In [1]:
#importing actual data
import pandas as pd
import numpy as np

#lmf file read at once remove d-type error
#latin1 for special characters oblskip for corrupted lines
df_pr = pd.read_csv("PR2025.txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')
#tx and la for more datapts
df_tx = pd.read_csv("TX25.txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')
#la louisiana not los
df_la = pd.read_csv("LA25.txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')

In [2]:
# rid of quantile-based high_risk target 
#phase 2!
#cleaning
# combine datasets
df_combine = pd.concat([df_pr, df_tx, df_la], ignore_index=True)
print("raw combined:", len(df_combine))
# def columns
condition_cols = ['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CHANNEL_COND_061']
numeric_cols = ['YEAR_BUILT_027', 'ADT_029', 'SCOUR_CRITICAL_113', 'PERCENT_ADT_TRUCK_109']
# convert to numeric
for col in condition_cols + numeric_cols:
    df_combine[col] = pd.to_numeric(df_combine[col], errors='coerce')
df_combine = df_combine.dropna(subset=condition_cols)
print("after dropping missing condition ratings:", len(df_combine))

#fill in missing values in numeric cols with median values
for col in numeric_cols:
    df_combine[col] = df_combine[col].fillna(df_combine[col].median())

# bridge age
#last time NBI updated 2025
df_combine['bridge_age'] = 2025 - df_combine['YEAR_BUILT_027']
df_combine = df_combine[df_combine['bridge_age'] > 0]

#  deterioration rates (O&O)
#eq (9-feat.)/age
df_combine['DetR_deck'] = (9 - df_combine['DECK_COND_058']) / df_combine['bridge_age']
df_combine['DetR_super'] = (9 - df_combine['SUPERSTRUCTURE_COND_059']) / df_combine['bridge_age']
df_combine['DetR_sub'] = (9 - df_combine['SUBSTRUCTURE_COND_060']) / df_combine['bridge_age']
df_combine['DetR_overall'] = df_combine[['DetR_deck', 'DetR_super', 'DetR_sub']].max(axis=1)

# remove infinity values
df_combine = df_combine.replace([np.inf, -np.inf], np.nan)
df_combine = df_combine.dropna(subset=['DetR_deck', 'DetR_super', 'DetR_sub'])

print("final combined:", len(df_combine))
print(df_combine['STATE_CODE_001'].value_counts())


raw combined: 72069
after dropping missing condition ratings: 35275
final combined: 35273
STATE_CODE_001
48    25094
22     8663
72     1516
Name: count, dtype: int64


In [3]:
# target variable using NBI standard threshold(4)
# bridge is high risk if ANY primary component rated 4 or below
# 4-= poor condition

df_combine['high_risk'] = ((df_combine['DECK_COND_058'] <= 4) |
    (df_combine['SUPERSTRUCTURE_COND_059'] <= 4) |
    (df_combine['SUBSTRUCTURE_COND_060'] <= 4)).astype(int)

print("class distribution:")
print(df_combine['high_risk'].value_counts())
print("high risk:", round(df_combine['high_risk'].mean()*100,5),"%")

class distribution:
high_risk
0    33229
1     2044
Name: count, dtype: int64
high risk: 5.7948 %


In [4]:
# UNCHANGED (x is now RAW, not z-scored -- see deleted standardization cell above)
# redef feats & target
features_clean = ['bridge_age','ADT_029', 'PERCENT_ADT_TRUCK_109',
                  'SCOUR_CRITICAL_113','CHANNEL_COND_061']
x = df_combine[features_clean]
y = df_combine['high_risk']

print("Class distribution:")
print(y.value_counts())
print("Percentage high risk:", round(y.mean()*100,5))

Class distribution:
high_risk
0    33229
1     2044
Name: count, dtype: int64
Percentage high risk: 5.7948


In [5]:
# replace old "SMOTETomek on full x,y" cell nd  old "64 16 20 split from oo" cell 
##split first resample training fold only, skl stuff
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.ensemble import RandomForestClassifier
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

# split real data first 64 16 20   on real class balance
X_temp, X_test, Y_temp, Y_test = train_test_split(x, y, test_size=0.20,random_state=42, stratify=y)
X_train, X_val, Y_train, Y_val = train_test_split(X_temp, Y_temp, test_size=0.20,random_state=42, stratify=Y_temp)

print("training set (real):", len(X_train))
print("validation set (real):", len(X_val))
print("test set (real):", len(X_test))

# smotetomek on the training fold, k=5 OO, 
# val/test are never touched by resampling
#seed smote too, otherwise numbers change every run
smote = SMOTETomek(smote=SMOTE(k_neighbors=5, random_state=42), random_state=42)
X_train_res, Y_train_res = smote.fit_resample(X_train, Y_train)
print("Training set after SMOTETomek:", pd.Series(Y_train_res).value_counts().to_dict())

# STEP 3: train RF on the resampled TRAINING data only
rf_model = RandomForestClassifier(n_estimators=100, bootstrap=True, random_state=42,
                                   class_weight='balanced')
rf_model.fit(X_train_res, Y_train_res)
print("Random Forest trained")

# STEP 4: Platt-scale calibration on the REAL (non-resampled) validation set
rf_calibrated = CalibratedClassifierCV(FrozenEstimator(rf_model), method='sigmoid')
rf_calibrated.fit(X_val, Y_val)
print("Platt scaling applied")

training set (real): 22574
validation set (real): 5644
test set (real): 7055
Training set after SMOTETomek: {0: 20730, 1: 20730}
Random Forest trained
Platt scaling applied


In [6]:
# model Validation
# oo multi-metric evaluation
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score, classification_report, balanced_accuracy_score, confusion_matrix
Y_pred = rf_calibrated.predict(X_test)
Y_prob = rf_calibrated.predict_proba(X_test)[:, 1]
cm = confusion_matrix(Y_test, Y_pred)
print("confusion Matrix:")
print(cm)
print("balanced Accuracy:", round(balanced_accuracy_score(Y_test, Y_pred),5))
print("matthews Correlation Coefficient:", round(matthews_corrcoef(Y_test, Y_pred),5))
print("cohen Kappa:", round(cohen_kappa_score(Y_test, Y_pred),5))
print("classification Report:")
print(classification_report(Y_test, Y_pred))

confusion Matrix:
[[6597   49]
 [ 378   31]]
balanced Accuracy: 0.53421
matthews Correlation Coefficient: 0.15101
cohen Kappa: 0.10991
classification Report:
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      6646
           1       0.39      0.08      0.13       409

    accuracy                           0.94      7055
   macro avg       0.67      0.53      0.55      7055
weighted avg       0.91      0.94      0.92      7055



In [7]:
# REPLACES the old cross-validation cell ,SMOTE now happens inside each fold
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_pipeline = ImbPipeline([('smote', SMOTETomek(smote=SMOTE(k_neighbors=5, random_state=42), random_state=42)),   #seeded smote
    ('rf', RandomForestClassifier(n_estimators=100, bootstrap=True, random_state=42, class_weight='balanced'))])

cv_results = cross_validate(cv_pipeline, x, y, cv=cv,
    scoring=['balanced_accuracy', 'matthews_corrcoef', 'f1'], return_train_score=True)

print("Cross Validation Results:")
for metric in ['balanced_accuracy', 'matthews_corrcoef', 'f1']:
    test_scores = cv_results[f'test_{metric}']
    train_scores = cv_results[f'train_{metric}']
    print(metric.upper())
    print("Test:",  round(test_scores.mean(),5), "±", round(test_scores.std(),5))
    print("Train:", round(train_scores.mean(),5), "±", round(train_scores.std(),5))

Cross Validation Results:
BALANCED_ACCURACY
Test: 0.62835 ± 0.01065
Train: 0.9712 ± 0.00195
MATTHEWS_CORRCOEF
Test: 0.28794 ± 0.0146
Train: 0.92995 ± 0.00245
F1
Test: 0.32171 ± 0.01622
Train: 0.93395 ± 0.00239


In [8]:
# UNCHANGED bc already used the real X_test/Y_test
from sklearn.utils import resample
bootstrap_scores = []

for i in range(1000):
    X_boot, Y_boot = resample(X_test, Y_test, random_state=i)
    Y_boot_pred = rf_calibrated.predict(X_boot)
    score = balanced_accuracy_score(Y_boot, Y_boot_pred)
    bootstrap_scores.append(score)

bootstrap_scores = np.array(bootstrap_scores)

ci_lower = np.percentile(bootstrap_scores, 2.5)
ci_upper = np.percentile(bootstrap_scores, 97.5)

print("bootstrap balanced accuracy:")
print("Mean:", round(bootstrap_scores.mean(),4))
print("95% CI:", round(ci_lower,4), round(ci_upper,4))

bootstrap balanced accuracy:
Mean: 0.5341
95% CI: 0.521 0.5477


In [9]:
# UNCHANGED
from sklearn.inspection import permutation_importance

baseline_importance = permutation_importance(rf_calibrated, X_test, Y_test,
    n_repeats=30,random_state=42, scoring='balanced_accuracy')

importance_df = pd.DataFrame({'Feature': features_clean,'Mean Importance':
                baseline_importance.importances_mean,'Std Dev': baseline_importance.importances_std
                }).sort_values('Mean Importance', ascending=False)

print("Baseline Permutation Importance Rankings:")
print(importance_df.round(4))

Baseline Permutation Importance Rankings:
                 Feature  Mean Importance  Std Dev
2  PERCENT_ADT_TRUCK_109           0.0244   0.0029
0             bridge_age           0.0214   0.0038
3     SCOUR_CRITICAL_113           0.0182   0.0027
1                ADT_029           0.0166   0.0036
4       CHANNEL_COND_061           0.0089   0.0042


In [10]:
# REPLACES: the PR16/TX16 print cell, both "deteriorated" cells, and both
# "PR18/TX18 had errors" reload cells -- one clean pass, using the fixed
# "(1)" files, that also computes bridge_age (missing before)
condition_cols = ['DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CHANNEL_COND_061']

PR_before = pd.read_csv("PR16.txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')
TX_before = pd.read_csv("TX16.txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')
PR_after  = pd.read_csv("PR18 (1).txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')
TX_after  = pd.read_csv("TX18 (1).txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')

for df in [PR_before, TX_before, PR_after, TX_after]:
    for col in condition_cols + ['YEAR_BUILT_027']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

pr_merged = PR_before.merge(PR_after[['STRUCTURE_NUMBER_008'] + condition_cols],
                             on='STRUCTURE_NUMBER_008', suffixes=('_before', '_after'))
tx_merged = TX_before.merge(TX_after[['STRUCTURE_NUMBER_008'] + condition_cols],
                             on='STRUCTURE_NUMBER_008', suffixes=('_before', '_after'))

for merged in [pr_merged, tx_merged]:
    merged['deteriorated'] = (
        (merged['DECK_COND_058_after'] < merged['DECK_COND_058_before']) |
        (merged['SUPERSTRUCTURE_COND_059_after'] < merged['SUPERSTRUCTURE_COND_059_before']) |
        (merged['SUBSTRUCTURE_COND_060_after'] < merged['SUBSTRUCTURE_COND_060_before']) |
        (merged['CHANNEL_COND_061_after'] < merged['CHANNEL_COND_061_before'])
    ).astype(int)
    # bridge_age at the 2016 "before" inspection (was 2025 - made every bridge 9 yrs too old)
    merged['bridge_age'] = 2016 - merged['YEAR_BUILT_027']

print("PR bridges deteriorated post-storm:", pr_merged['deteriorated'].sum(), "/", len(pr_merged))
print("TX bridges deteriorated post-storm:", tx_merged['deteriorated'].sum(), "/", len(tx_merged))

PR bridges deteriorated post-storm: 107 / 2305
TX bridges deteriorated post-storm: 9430 / 52897


In [11]:
# REPLACES: "Refit scaler..." cell (deleted -- no scaler needed) and the
# hurricane prediction cell -- now predicts on raw features directly, and
# adds the significance test the plan called for
pr_test_features = pr_merged[['bridge_age', 'ADT_029', 'PERCENT_ADT_TRUCK_109',
                               'SCOUR_CRITICAL_113', 'CHANNEL_COND_061_before']].copy()
pr_test_features.columns = features_clean

tx_test_features = tx_merged[['bridge_age', 'ADT_029', 'PERCENT_ADT_TRUCK_109',
                               'SCOUR_CRITICAL_113', 'CHANNEL_COND_061_before']].copy()
tx_test_features.columns = features_clean

for col in features_clean:
    pr_test_features[col] = pd.to_numeric(pr_test_features[col], errors='coerce')
    tx_test_features[col] = pd.to_numeric(tx_test_features[col], errors='coerce')

hurricane_test = pd.concat([pr_test_features, tx_test_features], ignore_index=True)
hurricane_labels = pd.concat([pr_merged['deteriorated'], tx_merged['deteriorated']], ignore_index=True)

valid_idx = hurricane_test.notna().all(axis=1)
hurricane_test = hurricane_test[valid_idx]
hurricane_labels = hurricane_labels[valid_idx]

print("Hurricane validation set size:", len(hurricane_test))
print("Deteriorated:", hurricane_labels.sum())

# predict directly -- raw features, matching raw training features, no scaler involved
hurricane_pred = rf_calibrated.predict(hurricane_test)
hurricane_probs = rf_calibrated.predict_proba(hurricane_test)[:, 1]

print("Balanced Accuracy on Hurricane Pre-Validation:",
      round(balanced_accuracy_score(hurricane_labels, hurricane_pred), 4))
print(confusion_matrix(hurricane_labels, hurricane_pred))

from scipy import stats
# Welch's two-sample t-test: were predicted risk probabilities higher for
# bridges that went on to actually deteriorate?
t_stat, p_val = stats.ttest_ind(hurricane_probs[hurricane_labels == 1],
                                 hurricane_probs[hurricane_labels == 0],
                                 equal_var=False)
print("t-statistic:", round(t_stat, 4), " p-value:", p_val)

Hurricane validation set size: 31266
Deteriorated: 4956
Balanced Accuracy on Hurricane Pre-Validation: 0.5002
[[26257    53]
 [ 4944    12]]
t-statistic: -6.1707  p-value: 7.149345217519522e-10


In [12]:
# NEW -- tune the classification threshold on the REAL validation set only.
# X_val / Y_val were never touched by SMOTE and never used for test or
# hurricane evaluation, so tuning here doesn't leak into any reported number.
from sklearn.metrics import balanced_accuracy_score

val_probs = rf_calibrated.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.99, 197)
val_balanced_accs = [balanced_accuracy_score(Y_val, (val_probs >= t).astype(int)) for t in thresholds]

best_idx = np.argmax(val_balanced_accs)
best_threshold = thresholds[best_idx]

print("Default threshold (0.5) validation balanced accuracy:",
      round(balanced_accuracy_score(Y_val, (val_probs >= 0.5).astype(int)), 4))
print("Tuned threshold:", round(best_threshold, 4))
print("Tuned threshold validation balanced accuracy:", round(val_balanced_accs[best_idx], 4))

Default threshold (0.5) validation balanced accuracy: 0.5476
Tuned threshold: 0.045
Tuned threshold validation balanced accuracy: 0.7266


In [13]:
# reeval real test set at the tuned threshold not 0.5
test_probs = rf_calibrated.predict_proba(X_test)[:, 1]
Y_pred_tuned = (test_probs >= best_threshold).astype(int)

print(" Test set: default 0.5 vs tuned threshold", round(best_threshold, 3))
print("Confusion Matrix (tuned):")
print(confusion_matrix(Y_test, Y_pred_tuned))
print("Balanced Accuracy (tuned):", round(balanced_accuracy_score(Y_test, Y_pred_tuned), 4))
print("Balanced Accuracy (default 0.5, from earlier cell):",
      round(balanced_accuracy_score(Y_test, Y_pred), 4))
print("MCC (tuned):", round(matthews_corrcoef(Y_test, Y_pred_tuned), 4))
print("Classification Report (tuned):")
print(classification_report(Y_test, Y_pred_tuned))

 Test set: default 0.5 vs tuned threshold 0.045
Confusion Matrix (tuned):
[[5353 1293]
 [ 157  252]]
Balanced Accuracy (tuned): 0.7108
Balanced Accuracy (default 0.5, from earlier cell): 0.5342
MCC (tuned): 0.2382
Classification Report (tuned):
              precision    recall  f1-score   support

           0       0.97      0.81      0.88      6646
           1       0.16      0.62      0.26       409

    accuracy                           0.79      7055
   macro avg       0.57      0.71      0.57      7055
weighted avg       0.92      0.79      0.84      7055



In [14]:
# NEW re-evaluate the hurricane pre-validation set at the SAME tuned
# threshold (fit on validation data only, never on the hurricane set itself)
hurricane_pred_tuned = (hurricane_probs >= best_threshold).astype(int)

print("=== Hurricane validation: default 0.5 vs tuned threshold", round(best_threshold, 3), "===")
print("Confusion Matrix (tuned):")
print(confusion_matrix(hurricane_labels, hurricane_pred_tuned))
print("Balanced Accuracy (tuned):",
      round(balanced_accuracy_score(hurricane_labels, hurricane_pred_tuned), 4))
print("Balanced Accuracy (default 0.5, from earlier cell):",
      round(balanced_accuracy_score(hurricane_labels, hurricane_pred), 4))

=== Hurricane validation: default 0.5 vs tuned threshold 0.045 ===
Confusion Matrix (tuned):
[[20115  6195]
 [ 4051   905]]
Balanced Accuracy (tuned): 0.4736
Balanced Accuracy (default 0.5, from earlier cell): 0.5002


In [15]:
# sanity check on final feature distributions
print(x[features_clean].describe())

         bridge_age        ADT_029  PERCENT_ADT_TRUCK_109  SCOUR_CRITICAL_113  \
count  35273.000000   35273.000000           35273.000000        35273.000000   
mean      42.289258    6616.940606               8.991013            6.789159   
std       25.250624   15449.591450               9.671821            1.660644   
min        1.000000       0.000000               0.000000            0.000000   
25%       21.000000     200.000000               2.000000            5.000000   
50%       40.000000    1391.000000               6.000000            8.000000   
75%       60.000000    6741.000000              13.000000            8.000000   
max      170.000000  517256.000000              99.000000            9.000000   

       CHANNEL_COND_061  
count      35273.000000  
mean           6.503671  
std            0.890809  
min            0.000000  
25%            6.000000  
50%            7.000000  
75%            7.000000  
max            9.000000  


In [16]:
# NEW check   hurricane t-test direction  in each
# state separately, or is a pooling artifact (PR ~4.6% post-storm deteriorationvs TX ~17.8%. very diff base rates pooled into one test)
from scipy import stats

n_pr_valid = int(valid_idx[:len(pr_test_features)].sum())
for label, sl in [("Puerto Rico", slice(0, n_pr_valid)), ("Texas", slice(n_pr_valid, None))]:
    probs_g = hurricane_probs[sl]
    labels_g = hurricane_labels.values[sl]
    t, p = stats.ttest_ind(probs_g[labels_g==1], probs_g[labels_g==0], equal_var=False)
    print(label, "n:", len(labels_g), "deteriorated:", int(labels_g.sum()), "mean risk (deteriorated):", 
          round(probs_g[labels_g==1].mean(), 4), "mean risk (not):", round(probs_g[labels_g==0].mean(), 4), "t:", round(t, 4),"p:", p)

Puerto Rico n: 1667 deteriorated: 79 mean risk (deteriorated): 0.0993 mean risk (not): 0.0744 t: 1.5877 p: 0.11621079189532518
Texas n: 29599 deteriorated: 4877 mean risk (deteriorated): 0.0449 mean risk (not): 0.0488 t: -5.6124 p: 2.0662993179374164e-08


In [17]:
# tropical holdout test -- guam + us virgin islands
# model was only trained on pr tx la so it has never seen either one
# usvi is in the 2025 all states file, guam isnt so pull it from the usdot arcgis map server (2023 data)
import requests

df_all = pd.read_csv("Downloads/2025HwyBridgesDelimitedAllStates.txt", encoding='latin-1', low_memory=False, on_bad_lines='skip')
df_all['STATE_CODE_001'] = pd.to_numeric(df_all['STATE_CODE_001'], errors='coerce')
df_vi = df_all[df_all['STATE_CODE_001'] == 78].copy()

url = "https://geo.dot.gov/server/rest/services/hosted/National_Bridge_Inventory/FeatureServer/0/query"
resp = requests.get(url, params={'where': "state_code='66'", 'outFields': '*',
                                 'returnGeometry': 'false', 'f': 'json'}, timeout=60).json()
df_gu = pd.DataFrame([f['attributes'] for f in resp['features']])
#arcgis cuts col names to 10 letters, rename back to nbi names
df_gu = df_gu.rename(columns={'year_built': 'YEAR_BUILT_027', 'adt_029': 'ADT_029',
                              'percent_ad': 'PERCENT_ADT_TRUCK_109', 'scour_crit': 'SCOUR_CRITICAL_113',
                              'channel_co': 'CHANNEL_COND_061', 'deck_cond_': 'DECK_COND_058',
                              'superstruc': 'SUPERSTRUCTURE_COND_059', 'substructu': 'SUBSTRUCTURE_COND_060'})
df_gu['STATE_CODE_001'] = 66
print("guam raw:", len(df_gu), " usvi raw:", len(df_vi))

#same cleaning as training
df_trop = pd.concat([df_gu, df_vi], ignore_index=True)
for col in condition_cols + numeric_cols:
    df_trop[col] = pd.to_numeric(df_trop[col], errors='coerce')
df_trop = df_trop.dropna(subset=condition_cols).copy()   #copy so pandas stops the fragmented warning
#fill w training medians not their own
for col in numeric_cols:
    df_trop[col] = df_trop[col].fillna(df_combine[col].median())
#age at the inspection year -- guam ratings are 2023
df_trop['bridge_age'] = np.where(df_trop['STATE_CODE_001'] == 66, 2023, 2025) - df_trop['YEAR_BUILT_027']
df_trop = df_trop[df_trop['bridge_age'] > 0]
df_trop['high_risk'] = ((df_trop['DECK_COND_058'] <= 4) | (df_trop['SUPERSTRUCTURE_COND_059'] <= 4) |
    (df_trop['SUBSTRUCTURE_COND_060'] <= 4)).astype(int)
print(df_trop.groupby('STATE_CODE_001')['high_risk'].agg(['size', 'sum']))

guam raw: 47  usvi raw: 24
                size  sum
STATE_CODE_001           
66                34    6
78                11    3


In [18]:
# score guam + usvi w the same calibrated model nd the same tuned threshold (no retraining)
from sklearn.metrics import roc_auc_score

x_trop = df_trop[features_clean]
y_trop = df_trop['high_risk']
trop_probs = rf_calibrated.predict_proba(x_trop)[:, 1]
trop_pred = (trop_probs >= best_threshold).astype(int)

#auc = chance a random high risk bridge gets a higher score than a random low risk one (0.5 = coin flip)
#using auc here bc it doesnt depend on the threshold or how many poor bridges a place has
print("tropical holdout n:", len(y_trop), " high risk:", int(y_trop.sum()))
print("auc:", round(roc_auc_score(y_trop, trop_probs), 4))
print("balanced accuracy (tuned):", round(balanced_accuracy_score(y_trop, trop_pred), 4))
print(confusion_matrix(y_trop, trop_pred))

#bootstrap ci bc n is tiny
boot_auc = []
for i in range(1000):
    pb, yb = resample(trop_probs, y_trop.values, random_state=i)
    if yb.sum() == 0 or yb.sum() == len(yb):
        continue
    boot_auc.append(roc_auc_score(yb, pb))
print("95% CI:", round(np.percentile(boot_auc, 2.5), 4), round(np.percentile(boot_auc, 97.5), 4))

#compare to the normal test set (pr tx la)
print("test set auc (pr tx la):", round(roc_auc_score(Y_test, test_probs), 4))

tropical holdout n: 45  high risk: 9
auc: 0.8056
balanced accuracy (tuned): 0.7778
[[28  8]
 [ 2  7]]
95% CI: 0.6368 0.9488
test set auc (pr tx la): 0.8024


In [19]:
# leave one state out (loso) -- every state in the 2025 nbi, not just pr tx la
# train on all the other states, test on the one left out
# shows how well the model moves to a place its never seen (harder than new bridges in the same states)
# no smote here: auc only cares about ranking + smote x50 states takes way too long
from sklearn.neighbors import NearestNeighbors

df_us = df_all.copy()
for col in condition_cols + numeric_cols:
    df_us[col] = pd.to_numeric(df_us[col], errors='coerce')
df_us = df_us.dropna(subset=condition_cols)
for col in numeric_cols:
    df_us[col] = df_us[col].fillna(df_us[col].median())
df_us['bridge_age'] = 2025 - df_us['YEAR_BUILT_027']
df_us = df_us[df_us['bridge_age'] > 0]
df_us['high_risk'] = ((df_us['DECK_COND_058'] <= 4) | (df_us['SUPERSTRUCTURE_COND_059'] <= 4) |
    (df_us['SUBSTRUCTURE_COND_060'] <= 4)).astype(int)
print("all states clean:", len(df_us), " high risk:", round(df_us['high_risk'].mean()*100, 3), "%")

#put feats on the same scale (log adt bc its super skewed)
def feat_space(df, ref):
    a = df[features_clean].copy()
    b = ref[features_clean].copy()
    a['ADT_029'] = np.log1p(a['ADT_029'])
    b['ADT_029'] = np.log1p(b['ADT_029'])
    return (a - b.mean()) / b.std()

#how far a place is from the training data: avg distance to its 10 nearest training bridges,
#divided by how far training bridges are from each other (meyer & pebesma 2021 idea). 1 = same spread
def dist_ratio(test, train):
    ref = train.sample(n=min(20000, len(train)), random_state=42)
    nn = NearestNeighbors(n_neighbors=11).fit(feat_space(ref, ref))
    t = test.sample(n=min(3000, len(test)), random_state=42)
    d_test = nn.kneighbors(feat_space(t, ref), n_neighbors=10)[0].mean()
    d_self = nn.kneighbors(feat_space(ref.iloc[:3000], ref))[0][:, 1:].mean()  #skip itself
    return d_test / d_self

#label free auc estimate (cbpe idea): if probs are calibrated then prob = chance its high risk,
#so the expected auc can be worked out w/o knowing any real labels
def est_auc(p):
    d = pd.DataFrame({'p': p})
    g = d.groupby('p')['p'].agg(['sum', 'count']).sort_index()
    pos = g['sum'].values
    neg = g['count'].values - pos
    below = np.cumsum(neg) - neg   #expected low risk bridges w a lower score
    same = (d['p'] * (1 - d['p'])).sum()
    num = (pos * below).sum() + 0.5 * ((pos * neg).sum() - same)   #ties count half
    den = d['p'].sum() * (1 - d['p']).sum() - same
    return num / den

#same rf + platt setup as before, training sample capped at 60k so 50 fits dont take forever
def fit_cal(train):
    train = train.sample(n=min(60000, len(train)), random_state=42)
    tr, cal = train_test_split(train, test_size=0.20, random_state=42, stratify=train['high_risk'])
    rf_s = RandomForestClassifier(n_estimators=100, bootstrap=True, random_state=42,
                                  class_weight='balanced', n_jobs=-1)
    rf_s.fit(tr[features_clean], tr['high_risk'])
    rf_s_cal = CalibratedClassifierCV(FrozenEstimator(rf_s), method='sigmoid')
    rf_s_cal.fit(cal[features_clean], cal['high_risk'])
    return rf_s_cal, tr

loso = []
for st in sorted(df_us['STATE_CODE_001'].unique()):
    test = df_us[df_us['STATE_CODE_001'] == st]
    if test['high_risk'].sum() < 20:   #too few poor bridges for a stable auc
        continue
    model_s, tr = fit_cal(df_us[df_us['STATE_CODE_001'] != st])
    p = model_s.predict_proba(test[features_clean])[:, 1]
    loso.append({'state': int(st), 'n': len(test), 'rate': test['high_risk'].mean(),
                 'auc': roc_auc_score(test['high_risk'], p), 'est_auc': est_auc(p),
                 'dist': dist_ratio(test, tr)})

loso = pd.DataFrame(loso)
loso['error'] = loso['est_auc'] - loso['auc']
print("states tested:", len(loso))
print(loso.sort_values('auc').round(3).to_string(index=False))
print("auc median:", round(loso['auc'].median(), 4), " min:", round(loso['auc'].min(), 4), " max:", round(loso['auc'].max(), 4))

all states clean: 363288  high risk: 9.178 %
states tested: 49
 state     n  rate   auc  est_auc  dist  error
     6 13282 0.069 0.629    0.668 1.378  0.038
    53  5903 0.062 0.633    0.642 1.478  0.009
    41  6666 0.059 0.654    0.685 1.262  0.032
    15   770 0.071 0.655    0.689 1.185  0.034
    12  6655 0.042 0.670    0.611 1.001 -0.059
     5  8366 0.074 0.676    0.617 0.926 -0.059
    25  2025 0.111 0.683    0.660 1.405 -0.023
    24  2384 0.062 0.689    0.648 1.053 -0.041
    51  7601 0.044 0.693    0.691 0.987 -0.002
     8  5430 0.057 0.700    0.655 1.242 -0.044
    48 25094 0.021 0.700    0.625 1.157 -0.075
    21  9680 0.094 0.701    0.685 0.772 -0.016
    49  1343 0.057 0.705    0.643 1.466 -0.062
    47  8425 0.075 0.706    0.666 0.829 -0.040
     2  1419 0.078 0.706    0.642 1.680 -0.065
    45  6999 0.071 0.710    0.658 0.931 -0.052
    17 17995 0.108 0.711    0.661 1.104 -0.050
     9  1697 0.069 0.713    0.664 0.975 -0.050
    35  1526 0.081 0.716    0.632 1.084 -0.0

In [20]:
# can we tell ahead of time (w/o any labels) how well the model will do in a new place?
# test 1: does distance from the training data predict the real auc?
# test 2: does the label free estimate match the real auc?
from scipy import stats

r_dist, p_dist = stats.spearmanr(loso['dist'], loso['auc'])
r_est, p_est = stats.spearmanr(loso['est_auc'], loso['auc'])
print("distance vs real auc   spearman:", round(r_dist, 3), " p:", round(p_dist, 4))
print("estimate vs real auc   spearman:", round(r_est, 3), " p:", round(p_est, 4))
print("estimate mean abs error:", round(loso['error'].abs().mean(), 4))
#is the estimate off more in states w a lot (or very few) poor bridges
r_rate, p_rate = stats.spearmanr(loso['rate'], loso['error'])
print("base rate vs estimate error   spearman:", round(r_rate, 3), " p:", round(p_rate, 4))

#error band for a brand new place = middle 90% of the 50 past errors (split conformal idea)
lo, hi = np.percentile(loso['error'], [5, 95])
print("90% error band (est - real):", round(lo, 4), "to", round(hi, 4))

# check the whole thing on guam + usvi where we DO know the answer
model_t, tr_t = fit_cal(df_us[df_us['STATE_CODE_001'] != 78])
p_t = model_t.predict_proba(x_trop)[:, 1]
est_t = est_auc(p_t)
print("guam+usvi distance ratio:", round(dist_ratio(df_trop, tr_t), 3),
      " (us states range", round(loso['dist'].min(), 3), "-", round(loso['dist'].max(), 3), ")")
print("guam+usvi estimated auc:", round(est_t, 4), " predicted range:", round(est_t - hi, 4), "-", round(est_t - lo, 4))
print("guam+usvi real auc:", round(roc_auc_score(y_trop, p_t), 4))
#pr is in the loso table too (state 72)
print(loso[loso['state'].isin([12, 15, 22, 48, 72])].round(3).to_string(index=False))

distance vs real auc   spearman: -0.226  p: 0.1184
estimate vs real auc   spearman: 0.384  p: 0.0065
estimate mean abs error: 0.0663
base rate vs estimate error   spearman: 0.077  p: 0.5983
90% error band (est - real): -0.1358 to 0.0226
guam+usvi distance ratio: 1.253  (us states range 0.764 - 1.68 )
guam+usvi estimated auc: 0.6413  predicted range: 0.6188 - 0.7772
guam+usvi real auc: 0.8642
 state     n  rate   auc  est_auc  dist  error
    12  6655 0.042 0.670    0.611 1.001 -0.059
    15   770 0.071 0.655    0.689 1.185  0.034
    22  8663 0.143 0.752    0.647 1.079 -0.105
    48 25094 0.021 0.700    0.625 1.157 -0.075
    72  1516 0.179 0.721    0.666 1.216 -0.055
